03_feature_engineering.ipynb
+ yeni feature'ların doğrulanması
+ feature importance analizleri
+ feature karşılaştırmaları

**Baseline modelimiz:**
ROC-AUC = 0.7724

>Bu sonucu hangi feature'lar üretti?

In [1]:
import sys
import os

sys.path.append(os.path.abspath(".."))

In [ ]:
import joblib
import pandas as pd

from src.preprocessing.prepare_dataset import (
    prepare_dataset
)

from src.models.feature_importance import (
    get_feature_importance
)

In [ ]:
df = prepare_dataset(
    "../data/processed/train_feature_store.parquet"
)

print(df.shape)

(307511, 149)


+ Önce : 150 kolon
+ Sonra : 149 kolon (SK_ID_CURR çıkarıldı)


In [ ]:
X = df.drop(columns=["TARGET"])

feature_names = X.columns

len(feature_names)



148

In [ ]:
model = joblib.load(
    "../artifacts/models/lgbm_baseline.pkl"
)

importance_df = get_feature_importance(
    model,
    feature_names
)

importance_df.head(20)

,feature,importance
39,EXT_SOURCE_1,462
40,EXT_SOURCE_2,419
41,EXT_SOURCE_3,410
122,credit_term,308
130,annuity_credit_ratio,305
18,DAYS_ID_PUBLISH,269
15,DAYS_BIRTH,240
134,bureau_total_credit,237
17,DAYS_REGISTRATION,232
139,bureau_debt_credit_ratio,206


## 📊 Öne Çıkan Özellikler (Feature Importance)

Model eğitimi sonucunda, kendi ürettiğimiz (engineered) özellikler ve harici veri kaynaklarından türetilen agregasyonlar arasında en yüksek ayırt edici güce sahip, **ilk 20'ye giren kritik özellikler** aşağıda listelenmiştir:

|  Sıra  | Özellik Adı (Feature Name)    | Açıklama                                                                  |
| :----: | :---------------------------- | :------------------------------------------------------------------------ |
|  **1** | `credit_term`                 | Kredi tutarının taksit tutarına oranı; kredi vadesi hakkında bilgi sağlar |
|  **2** | `annuity_credit_ratio`        | Taksit tutarının toplam kredi tutarına oranı                              |
|  **3** | `bureau_total_credit`         | Kredi Kayıt Bürosu'ndaki toplam kredi hacmi                               |
|  **4** | `bureau_debt_credit_ratio`    | Toplam borcun toplam krediye oranı                                        |
|  **5** | `bureau_total_debt`           | Kredi Kayıt Bürosu'ndaki toplam mevcut borç                               |
|  **6** | `age_years`                   | Müşterinin yaşı (yıl bazında)                                             |
|  **7** | `credit_income_ratio`         | Kredi tutarının müşterinin gelirine oranı                                 |
|  **8** | `annuity_income_ratio`        | Taksit tutarının müşterinin gelirine oranı                                |
|  **9** | `prev_avg_credit_amount`      | Önceki başvurulardaki ortalama kredi tutarı                               |
| **10** | `prev_avg_application_amount` | Önceki başvurularda talep edilen ortalama kredi miktarı                   |
| **11** | `refusal_rate`                | Geçmiş kredi başvurularında reddedilme oranı                              |

### Sonuç

Feature engineering çalışmaları sonucunda oluşturulan değişkenlerin önemli bir bölümü modelin en etkili değişkenleri arasına girmiştir. Özellikle:

* Gelir–kredi ilişkisini ölçen oranlar (`credit_income_ratio`, `annuity_income_ratio`)
* Kredi geçmişini özetleyen bureau agregasyonları (`bureau_total_credit`, `bureau_total_debt`, `bureau_debt_credit_ratio`)
* Geçmiş kredi başvuru davranışlarını özetleyen değişkenler (`prev_avg_credit_amount`, `prev_avg_application_amount`, `refusal_rate`)

model tarafından güçlü risk sinyalleri olarak kullanılmıştır.

Bu sonuçlar, yalnızca ham başvuru verilerinin değil, müşteri geçmişinden türetilen davranışsal ve finansal göstergelerin de kredi temerrüt tahmininde önemli katkı sağladığını göstermektedir.


In [ ]:
importance_df[
    importance_df["feature"] == "SK_ID_CURR"
]

,feature,importance


In [ ]:
importance_df[
    importance_df["feature"] == "EXT_SOURCE_1"
]

,feature,importance
39,EXT_SOURCE_1,462


In [1]:
import pandas as pd

pos = pd.read_csv(
    "../data/raw/POS_CASH_balance.csv"
)

print(pos.shape)

(10001358, 8)


In [4]:
from src.features.build_pos_cash_features import (
    build_pos_cash_features
)

pos_features = (
    build_pos_cash_features(pos)
)

print(pos_features.shape)

(337252, 9)


**previous_features**    → 338857 müşteri

**installment_features** → 339587 müşteri

**bureau_features**    → 305811 müşteri

**pos_features**       → 337252 müşteri

In [5]:
pos_features.head()

,SK_ID_CURR,pos_record_count,pos_avg_dpd,pos_max_dpd,pos_avg_dpd_def,pos_max_dpd_def,pos_active_contracts,pos_completed_contracts,pos_avg_future_installments
0,100001,9,0.777778,7,0.777778,7,7.0,2.0,1.444444
1,100002,19,0.000000,0,0.000000,0,19.0,NaN,15.000000
2,100003,28,0.000000,0,0.000000,0,26.0,2.0,5.785714
3,100004,4,0.000000,0,0.000000,0,3.0,1.0,2.250000
4,100005,11,0.000000,0,0.000000,0,9.0,1.0,7.200000


In [6]:
pos_features.describe().T

,count,mean,std,min,25%,50%,75%,max
SK_ID_CURR,337252.0,278163.132678,102877.889290,100001.0,189046.75,278241.500000,367320.250000,456255.000000
pos_record_count,337252.0,29.655445,24.531971,1.0,12.00,22.000000,39.000000,295.000000
pos_avg_dpd,337252.0,4.296271,59.717229,0.0,0.00,0.000000,0.000000,2622.078431
pos_max_dpd,337252.0,15.294106,151.343806,0.0,0.00,0.000000,0.000000,4231.000000
pos_avg_dpd_def,337252.0,0.225470,13.554576,0.0,0.00,0.000000,0.000000,1740.554455
pos_max_dpd_def,337252.0,1.473355,32.337266,0.0,0.00,0.000000,0.000000,3595.000000
pos_active_contracts,337034.0,27.151916,22.723824,1.0,11.00,20.000000,36.000000,271.000000
pos_completed_contracts,300240.0,2.480959,3.253642,1.0,1.00,2.000000,3.000000,86.000000
pos_avg_future_installments,337224.0,9.176876,6.501034,0.0,5.00,6.989411,11.666667,60.000000


In [2]:
import joblib

from src.preprocessing.prepare_dataset import (
    prepare_dataset
)

from src.models.feature_importance import (
    get_feature_importance
)

df = prepare_dataset(
    "../data/processed/train_feature_store.parquet"
)

X = df.drop(columns=["TARGET"])

feature_names = X.columns

model = joblib.load(
    "../artifacts/models/lgbm_baseline.pkl"
)

importance_df = get_feature_importance(
    model,
    feature_names
)

importance_df.head(30)

,feature,importance
39,EXT_SOURCE_1,398
40,EXT_SOURCE_2,356
41,EXT_SOURCE_3,352
162,pos_avg_future_installments,344
130,annuity_credit_ratio,265
122,credit_term,231
18,DAYS_ID_PUBLISH,224
154,total_payment_amount,223
15,DAYS_BIRTH,209
139,bureau_debt_credit_ratio,200


### Ürettiğimiz Feature'lardan İlk 30'a Girenler
**Application**
Feature | Importance |
-----------|----------------|
annuity_credit_ratio|	265
credit_term	|231
annuity_income_ratio	|155
age_years	|125
credit_income_ratio|	110



**Bureau**
Feature|Importance |
-----------|---------------|
bureau_debt_credit_ratio|	200
bureau_total_credit	|191
bureau_total_debt	|144
bureau_active_loans	|139

**Previous Application**
Feature|Importance |
-----------|---------------|
prev_avg_application_amount|	112

**Installments**
Feature|Importance |
-----------|---------------|
total_payment_amount|	223
late_payment_ratio	|166
installment_count	|131
avg_payment_ratio	|130
avg_days_late  |	111

**POS Cash**
Feature	 | Importance
-----------|---------------|
pos_avg_future_installments|	344
pos_active_contracts|	122

## Feature Engineering Impact

Feature engineering çalışmaları sonucunda oluşturulan müşteri davranışı ve kredi geçmişi özellikleri model performansını önemli ölçüde artırmıştır.

Özellikle aşağıdaki özellikler en yüksek önem skorlarına ulaşmıştır:

- pos_avg_future_installments
- annuity_credit_ratio
- credit_term
- total_payment_amount
- bureau_debt_credit_ratio
- bureau_total_credit
- late_payment_ratio
- annuity_income_ratio
- pos_active_contracts
- avg_payment_ratio

Bu özellikler müşterinin mevcut yükümlülük seviyesini, ödeme disiplinini ve geçmiş kredi davranışını temsil etmektedir.